In [1]:
import pandas as pd
import numpy as np
import os

print("Memulai proses Data Integration dan Preprocessing...")

# 1. Membaca Kelima File CSV dari folder raw
print("Membaca raw data...")
telemetry = pd.read_csv('../data/raw/PdM_telemetry.csv')
errors = pd.read_csv('../data/raw/PdM_errors.csv')
maint = pd.read_csv('../data/raw/PdM_maint.csv')
failures = pd.read_csv('../data/raw/PdM_failures.csv')
machines = pd.read_csv('../data/raw/PdM_machines.csv')

Memulai proses Data Integration dan Preprocessing...
Membaca raw data...


In [4]:
# 2. Standarisasi Format Waktu (Krusial untuk Time-Series)
print("Standarisasi format datetime...")
for df in [telemetry, errors, maint, failures]:
    df['datetime'] = pd.to_datetime(df['datetime'])

Standarisasi format datetime...


In [6]:
# 3. Proses Penggabungan (Data Integration)
print("Menggabungkan data tabel...")
df_master = telemetry.copy()

df_master = df_master.merge(machines, on='machineID', how='left')

errors = errors.rename(columns={'errorID': 'error_type'})
df_master = df_master.merge(errors, on=['machineID', 'datetime'], how='left')

failures = failures.rename(columns={'failure': 'failure_type'})
df_master = df_master.merge(failures, on=['machineID', 'datetime'], how='left')

Menggabungkan data tabel...


In [7]:
# 4. Penanganan Nilai Kosong (Missing Values)
df_master['error_type'] = df_master['error_type'].fillna('None')
df_master['failure_type'] = df_master['failure_type'].fillna('None')

In [8]:
# 5. Feature Engineering: Rolling Window 24 Jam
print("Melakukan Feature Engineering (Pembuatan pola historis 24 jam)...")

df_master = df_master.sort_values(['machineID', 'datetime'])

sensor_cols = ['volt', 'rotate', 'pressure', 'vibration']

for col in sensor_cols:
    df_master[f'{col}_mean_24h'] = df_master.groupby('machineID')[col].transform(
        lambda x: x.rolling(window=24).mean()
    )
    df_master[f'{col}_std_24h'] = df_master.groupby('machineID')[col].transform(
        lambda x: x.rolling(window=24).std()
    )

Melakukan Feature Engineering (Pembuatan pola historis 24 jam)...


In [9]:
# 6. Labeling Target (Mendefinisikan X dan Y)
print("Membuat label target klasifikasi...")
# 1 = Mesin Rusak, 0 = Mesin Normal
df_master['is_broken'] = df_master['failure_type'].apply(lambda x: 0 if x == 'None' else 1)

Membuat label target klasifikasi...


In [10]:
# 7. Membuang nilai kosong akibat efek 'rolling'
# (23 jam pertama tiap mesin akan menjadi NaN karena belum cukup 24 jam data)
df_master = df_master.dropna()

print(f"Data siap ditraining! Ukuran akhir dataset: {df_master.shape}")

Data siap ditraining! Ukuran akhir dataset: (874145, 19)


In [11]:
# 8. Menyimpan Hasil Akhir
os.makedirs('../data/processed', exist_ok=True)
lokasi_simpan = '../data/processed/features_ready.csv'
df_master.to_csv(lokasi_simpan, index=False)

print(f"Data master dengan fitur lengkap berhasil disimpan di: {lokasi_simpan}")

Data master dengan fitur lengkap berhasil disimpan di: ../data/processed/features_ready.csv
